In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
from collections import defaultdict
from surprise import SVD
from sklearn.metrics.pairwise import cosine_similarity

import sys
sys.path.append('..')

from pipeline import Pipeline
from modules.adaptive.filters.rule_based import RuleBasedFiltering
from modules.adaptive.filters.collaborative import CollaborativeFiltering
from modules.adaptive.filters.content_based import ContentBasedFiltering
from modules.demographic_recommender import DemographicRecommender
from modules.hybrid import HybridContentProfileRecommender
from modules.hybrid_collaborating import HybridDemographicCollaborativeRecommender
from modules.personalization.recommender import Recommender
from modules.personalization.diversifier import Diversifier

# Helper Functions
def ndcg_at_k(y_true, y_pred, k=None):
    if k is None:
        k = len(y_true)
    
    sorted_indices = np.argsort(-np.array(y_pred))[:k]
    y_true_sorted = np.array(y_true)[sorted_indices]
    
    dcg = np.sum((2 ** y_true_sorted - 1) / np.log2(np.arange(1, len(y_true_sorted) + 1) + 1))
    
    ideal_sorted = np.sort(y_true)[::-1][:k]
    idcg = np.sum((2 ** ideal_sorted - 1) / np.log2(np.arange(1, len(ideal_sorted) + 1) + 1))
    
    return dcg / idcg if idcg > 0 else 0.0

def mean_reciprocal_rank(y_true, y_pred):
    sorted_indices = np.argsort(-np.array(y_pred))
    ranks = [rank + 1 for rank, index in enumerate(sorted_indices) if y_true[index] > 0]
    return 1 / ranks[0] if ranks else 0.0

def precision_at_k(y_true, y_pred, k):
    sorted_indices = np.argsort(-np.array(y_pred))[:k]
    y_true_sorted = np.array(y_true)[sorted_indices]
    return np.sum(y_true_sorted > 0) / k

def recall_at_k(y_true, y_pred, k):
    sorted_indices = np.argsort(-np.array(y_pred))[:k]
    y_true_sorted = np.array(y_true)[sorted_indices]
    return np.sum(y_true_sorted > 0) / np.sum(np.array(y_true) > 0)

def calculate_coverage(recommender, test_df, catalog_items):
    recommended_items = set()
    grouped = test_df.groupby("user_id")
    
    for user_id, group in grouped:
        recommendations = [recommender.predict(user_id, item_id) for item_id in catalog_items]
        top_items = np.argsort(-np.array(recommendations))[:10]
        recommended_items.update(catalog_items[top_items])
    
    return len(recommended_items) / len(catalog_items)

def calculate_diversity(recommender, test_df, similarity_matrix, k=10):
    diversity_scores = []
    grouped = test_df.groupby("user_id")
    
    for user_id, group in grouped:
        recommendations = [recommender.predict(user_id, item_id) for item_id in group["movie_id"]]
        top_items = np.argsort(-np.array(recommendations))[:k]
        pairwise_similarities = similarity_matrix[np.ix_(top_items, top_items)]
        diversity = 1 - np.mean(pairwise_similarities)
        diversity_scores.append(diversity)
    
    return np.mean(diversity_scores)

def evaluate_model(model, test_df, blending_weight=0.5, k=10):
    true_ratings = []
    predicted_ratings = []
    grouped = test_df.groupby("user_id")
    catalog_items = test_df["movie_id"].unique()
    item_features = test_df.drop(columns=["movie_id"]).values
    similarity_matrix = cosine_similarity(item_features)

    # recommender = Recommender(model=model, blending_weight=blending_weight)

    for _, group in grouped:
        for _, row in group.iterrows():
            user_id = row["user_id"]
            item_id = row["movie_id"]
            true_rating = row["rating"]
            
            predicted_rating = model.predict(user_id, item_id)
            # rankings = recommender.rank_items(user_id, top_n=10)
            true_ratings.append(true_rating)
            predicted_ratings.append(predicted_rating)
    
    if any(pd.isna(predicted_ratings)):
        nan_in_predicted = [i for i, val in enumerate(predicted_ratings) if pd.isna(val)]
        print(f"NaN in predicted_ratings at indices: {nan_in_predicted}")
        predicted_ratings = [0 if pd.isna(val) else val for val in predicted_ratings]

    true_ratings = true_ratings[:k]
    predicted_ratings = predicted_ratings[:k]
    
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(true_ratings, predicted_ratings))
    
    # Compute MAE
    mae = mean_absolute_error(true_ratings, predicted_ratings)
    
    # Compute nDCG
    ndcg_scores = []
    for user_id, group in grouped:
        y_true = group["rating"].tolist()
        y_pred = [model.predict(user_id, item_id) for item_id in group["movie_id"]]
        ndcg_scores.append(ndcg_at_k(y_true, y_pred, k))
    
    ndcg = np.mean(ndcg_scores)
    
    # Compute MRR
    mrr_scores = []
    for user_id, group in grouped:
        y_true = group["rating"].tolist()
        y_pred = [model.predict(user_id, item_id) for item_id in group["movie_id"]]
        mrr_scores.append(mean_reciprocal_rank(y_true, y_pred))
    
    mrr = np.mean(mrr_scores)
    
    # Compute Precision@K and Recall@K
    precision_scores = []
    recall_scores = []
    for user_id, group in grouped:
        y_true = group["rating"].tolist()
        y_pred = [model.predict(user_id, item_id) for item_id in group["movie_id"]]
        precision_scores.append(precision_at_k(y_true, y_pred, k))
        recall_scores.append(recall_at_k(y_true, y_pred, k))
    
    precision = np.mean(precision_scores)
    recall = np.mean(recall_scores)
    
    # Compute Coverage
    coverage = calculate_coverage(model, test_df, catalog_items)
    
    # Compute Diversity
    diversity = calculate_diversity(model, test_df, similarity_matrix, k)
    
    return {
        "RMSE": rmse,
        "MAE": mae,
        "nDCG": ndcg,
        "MRR": mrr,
        "Precision@K": precision,
        "Recall@K": recall,
        "Coverage": coverage,
        "Diversity": diversity
    }

class BaselineRecommender:
    def __init__(self, fixed_rating=3):
        self.fixed_rating = fixed_rating

    def predict(self, user_id, item_id):
        return self.fixed_rating

In [2]:
pipeline = Pipeline()
ratings_df = pipeline.load_data('../storage/u.data') 

Dataset loaded successfully.
Dataset Shape: (100000, 4)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
None

First few rows:
   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [9]:
train_df, test_df = pipeline.partition_data(ratings_df, partition_type='temporal', save_path='../storage', data_name='temporal')
metadata_file='../storage/u.item'
user_file='../storage/u.user'
ratings_file='../storage/train/stratified'

svd_params = {'n_factors': 200, 'n_epochs': 50, 'lr_all': 0.001, 'reg_all': 0.1}
collaborative_recommender = CollaborativeFiltering(
        ratings_file=ratings_file,
        metadata_file=metadata_file,
        algorithm=SVD(**svd_params)
    )
collaborative_recommender.fit()

k_values = [1, 5, 10, 20, 50, 100]
results = []
for k in k_values:
    result = evaluate_model(collaborative_recommender, test_df, k=k)
    results.append(result)
    print(f'Performance for {k} top recommendations of the Temporal Partitioned Data:', result)

Performance for 1 top recommendations of the Temporal Partitioned Data: {'RMSE': 0.9469997127623735, 'MAE': 0.9469997127623735, 'nDCG': 0.8147843083219783, 'MRR': 1.0, 'Precision@K': 1.0, 'Recall@K': 0.09230302197042983, 'Coverage': 0.04558011049723757, 'Diversity': 2.950758869435632e-17}
Performance for 5 top recommendations of the Temporal Partitioned Data: {'RMSE': 0.9443719641825538, 'MAE': 0.8407177884264773, 'nDCG': 0.8124020782413497, 'MRR': 1.0, 'Precision@K': 0.945514950166113, 'Recall@K': 0.2762991630082954, 'Coverage': 0.04558011049723757, 'Diversity': 2.7712420767163418e-14}
Performance for 10 top recommendations of the Temporal Partitioned Data: {'RMSE': 1.2002650701330453, 'MAE': 1.0107105463835064, 'nDCG': 0.8217315214833336, 'MRR': 1.0, 'Precision@K': 0.8953488372093024, 'Recall@K': 0.40502146581514575, 'Coverage': 0.04558011049723757, 'Diversity': 3.203823326975605e-14}
Performance for 20 top recommendations of the Temporal Partitioned Data: {'RMSE': 1.31001381720851, 

In [3]:
train_df, test_df = pipeline.partition_data(ratings_df, partition_type='temporal', save_path='../storage', data_name='temporal')
metadata_file='../storage/u.item'
user_file='../storage/u.user'
ratings_file='../storage/train/stratified'

weights = [0, 1]
models = {
    f"hybrid_recommender_weight{weight}": HybridDemographicCollaborativeRecommender(
        ratings_file=ratings_file,
        user_file=user_file,
        blending_weight=weight
    )
    for weight in weights
}

results = []
print(len(models))
for name, model in models.items():
    model.fit()
    result = evaluate_model(model, test_df)
    results.append(result)
    print(f'Performance for {name} of the Temporal Partitioned Data:', result)

1
Performance for hybrid_recommender_weight0 of the Temporal Partitioned Data: {'RMSE': 0.8195697804994854, 'MAE': 0.6457681641668482, 'nDCG': 0.8683997375104676, 'MRR': 1.0, 'Precision@K': 0.8953488372093024, 'Recall@K': 0.40502146581514575, 'Coverage': 0.11395027624309392, 'Diversity': 3.360877467801317e-14}


In [4]:
train_df, test_df = pipeline.partition_data(ratings_df, partition_type='temporal', save_path='../storage', data_name='temporal')
metadata_file='../storage/u.item'
# ratings_file='../storage/train/temporal'

# svd_params = {'n_factors': 200, 'n_epochs': 100, 'lr_all': 0.01, 'reg_all': 0.1}
# collaborative_recommender = CollaborativeFiltering(
#         ratings_file=ratings_file,
#         metadata_file=metadata_file,
#         algorithm=SVD(**svd_params)
#     )
# collaborative_recommender.fit()

# models = {
#     'RuleBasedFiltering': RuleBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
#     'ContentBasedFiltering': ContentBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
#     'CollaborativeFiltering': collaborative_recommender
# }

# metadata_file='../storage/u.item'
user_file='../storage/u.user'
ratings_file='../storage/train/stratified'

demographic_recommender = DemographicRecommender(
        ratings_file=ratings_file,
        user_file=user_file,
        item_file=metadata_file,
)
demographic_recommender.fit()

hybrid_recommender = HybridContentProfileRecommender(
        ratings_file=ratings_file,
        user_file=user_file,
        item_file=metadata_file,
)

svd_params = {'n_factors': 200, 'n_epochs': 100, 'lr_all': 0.01, 'reg_all': 0.1}
collaborative_recommender = CollaborativeFiltering(
        ratings_file=ratings_file,
        metadata_file=metadata_file,
        algorithm=SVD(**svd_params)
    )
collaborative_recommender.fit()

models = {
    'BaselineRecommender': BaselineRecommender(fixed_rating=3),
    'RuleBasedFiltering': RuleBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
    'ContentBasedFiltering': ContentBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
    'HybridRecommender': hybrid_recommender,
    'DemographicRecommender': demographic_recommender,
    'CollaborativeFiltering': collaborative_recommender
}

results = []

for name, model in models.items():
    result = evaluate_model(model, test_df)
    results.append(result)
    print(f'Performance for {name} of the Temporal Partitioned Data:', result)


Performance for BaselineRecommender of the Temporal Partitioned Data: {'RMSE': 1.2584116973391497, 'MAE': 1.0152, 'nDCG': 0.6129952146010054, 'MRR': 1.0, 'Precision@K': 0.8953488372093024, 'Recall@K': 0.40502146581514575, 'Coverage': 0.006906077348066298, 'Diversity': 2.1651930894201308e-14}
Performance for RuleBasedFiltering of the Temporal Partitioned Data: {'RMSE': 1.6933150425790326, 'MAE': 1.3292446338061923, 'nDCG': 0.6244722195649269, 'MRR': 1.0, 'Precision@K': 0.8953488372093024, 'Recall@K': 0.40502146581514575, 'Coverage': 0.04696132596685083, 'Diversity': 2.2875389290441057e-14}
Performance for ContentBasedFiltering of the Temporal Partitioned Data: {'RMSE': 3.237970723088892, 'MAE': 3.0341767594435214, 'nDCG': 0.6370632691720362, 'MRR': 1.0, 'Precision@K': 0.8953488372093024, 'Recall@K': 0.40502146581514575, 'Coverage': 0.30248618784530384, 'Diversity': 3.4421339901684005e-14}
Performance for HybridRecommender of the Temporal Partitioned Data: {'RMSE': 1.2614973004498382, 'M

In [5]:
train_df, test_df = pipeline.partition_data(ratings_df, partition_type='stratified', save_path='../storage', data_name='stratified')
metadata_file='../storage/u.item'
ratings_file='../storage/train/stratified'

svd_params = {'n_factors': 200, 'n_epochs': 100, 'lr_all': 0.01, 'reg_all': 0.1}
collaborative_recommender = CollaborativeFiltering(
        ratings_file=ratings_file,
        metadata_file=metadata_file,
        algorithm=SVD(**svd_params)
    )
collaborative_recommender.fit()

models = {
    'BaselineRecommender': BaselineRecommender(fixed_rating=3),
    'RuleBasedFiltering': RuleBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
    'ContentBasedFiltering': ContentBasedFiltering(ratings_file=ratings_file, metadata_file=metadata_file),
    'CollaborativeFiltering': collaborative_recommender,
    'HybridRecommender': hybrid_recommender,
    'DemographicRecommender': demographic_recommender,
}

results = []

for name, model in models.items():
    result = evaluate_model(model, test_df)
    results.append(result)
    print(f'Performance for {name} of the Stratified Partitioned Data:', result)

Performance for BaselineRecommender of the Stratified Partitioned Data: {'RMSE': 1.2403943558378832, 'MAE': 1.0005955926146515, 'nDCG': 0.4942716652297132, 'MRR': 1.0, 'Precision@K': 1.0, 'Recall@K': 0.19814316260647483, 'Coverage': 0.007547169811320755, 'Diversity': 7.13009898037045e-15}
Performance for RuleBasedFiltering of the Stratified Partitioned Data: {'RMSE': 1.090555495681426, 'MAE': 0.9088899226537356, 'nDCG': 0.5894531413342926, 'MRR': 1.0, 'Precision@K': 1.0, 'Recall@K': 0.19814316260647483, 'Coverage': 0.007547169811320755, 'Diversity': 8.63683022807602e-15}


KeyboardInterrupt: 